# Stage 1 Labeling Pipeline

这个 notebook 用于 Stage 1 事件级标签构建，覆盖 3 个核心步骤：

1. `rule pre-tag`
2. `LLM labeling`
3. `event matching and labeling prep`

主输出包括：

- `llm_labeling_response_template_v1.csv`
- `labels_mvp_v1.csv`
- `label_qc_audit_v1.csv`

推荐执行顺序：先生成规则预标、事件匹配和 LLM 输入，再完成 pilot、全量回填和最终合并。


In [13]:
from pathlib import Path
import json
import os
import random
import re
import time
from typing import Any

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

CANDIDATE_PATH = ROOT / 'data' / 'processed' / 'standardized' / '03_candidate_events_standardized.csv'
AUDIT_LIST_PATH = ROOT / 'data' / 'manual' / 'key_event_audit_list_v1.csv'
CANONICAL_PATH = ROOT / 'data' / 'manual' / 'key_event_match_canonical_v1.csv'
MATCH_CANDIDATES_PATH = ROOT / 'data' / 'manual' / 'key_event_match_candidates_v6.csv'
OVERRIDES_PATH = ROOT / 'data' / 'manual' / 'key_event_match_overrides_v1.csv'

RULE_OUTPUT_PATH = ROOT / 'data' / 'processed' / 'standardized' / '03_candidate_events_rule_pretag_v1.csv'
AUDIT_OUTPUT_PATH = ROOT / 'data' / 'manual' / 'key_event_audit_matches_v1.csv'
AUDIT_COVERAGE_PATH = ROOT / 'data' / 'manual' / 'key_event_audit_coverage_v1.csv'
LLM_INPUT_CSV_PATH = ROOT / 'data' / 'processed' / 'standardized' / '03_candidate_events_labeling_input_v1.csv'
LLM_BATCH_JSONL_PATH = ROOT / 'data' / 'manual' / 'llm_labeling_batch_input_v1.jsonl'
LLM_TEMPLATE_PATH = ROOT / 'data' / 'manual' / 'llm_labeling_response_template_v1.csv'
REVIEW_QUEUE_PATH = ROOT / 'data' / 'manual' / 'label_review_queue_v1.csv'
AUTO_PASS_SAMPLE_PATH = ROOT / 'data' / 'manual' / 'label_auto_pass_sample_v1.csv'
FINAL_LABEL_PATH = ROOT / 'data' / 'processed' / 'standardized' / 'labels_mvp_v1.csv'
QC_AUDIT_PATH = ROOT / 'data' / 'manual' / 'label_qc_audit_v1.csv'

candidate_df = pd.read_csv(CANDIDATE_PATH)
audit_df = pd.read_csv(AUDIT_LIST_PATH)
canonical_df = pd.read_csv(CANONICAL_PATH)
match_candidates_df = pd.read_csv(MATCH_CANDIDATES_PATH)
overrides_df = pd.read_csv(OVERRIDES_PATH)

print(f'candidate rows: {len(candidate_df):,}')
print(f'audit rows: {len(audit_df):,}')
print(f'core_top10 rows: {(audit_df["audit_tier"] == "core_top10").sum():,}')
print(f'canonical matched core_top10 rows: {len(canonical_df):,}')

audit_df.loc[audit_df['audit_tier'] == 'core_top10', ['key_event_id', 'event_name', 'expected_direction']]


candidate rows: 1,886
audit rows: 15
core_top10 rows: 10
canonical matched core_top10 rows: 10


,key_event_id,event_name,expected_direction
0,K01,NAFTA termination threat shifts to renegotiation,retreat
2,K03,ZTE export ban softening,retreat
4,K05,EU auto tariff threat paused after Juncker mee...,retreat
5,K06,G20 China tariff increase delayed by 90-day truce,retreat
6,K07,Mexico tariff threat suspended after border deal,retreat
8,K09,Phase One deal cancels December tariff round,retreat
11,K12,Heavy truck tariff imposed and maintained,non_retreat
12,K13,Mexico tariff pause after Sheinbaum border tro...,retreat
13,K14,Canada tariff pause after 30-day border deal,retreat
14,K15,China export controls and critical software cu...,non_retreat


## 1. Rule Pre-tag

这里用透明规则做高召回预标，不直接代替人工和 LLM。输出会保留：

- `rule_label`
- `rule_review_flag`
- `rule_review_reason`
- `retreat / pressure / negation` 命中明细


In [2]:
RETREAT_PATTERNS = {
    r'\bdelay(?:ed|ing|s)?\b': 3,
    r'\bpaus(?:e|ed|ing)\b': 3,
    r'\bsuspend(?:ed|ing|s)?\b': 3,
    r'\bexempt(?:ion|ions)?\b': 2,
    r'\bwaiver(?:s)?\b': 2,
    r'\btruce\b': 3,
    r'\brenegotiat(?:e|es|ed|ion|ing)\b': 3,
    r'\bagree(?:d|s)?\b': 2,
    r'\bstrong understanding\b': 3,
    r'\bno tariffs\b': 4,
    r'\bdrop all tariffs\b': 4,
    r'\bremove tariffs?\b': 4,
    r'get back into business': 4,
    r'\b90 day\b': 3,
    r'\bdeal\b': 1,
    r'\bmeeting\b': 1,
    r'\bnegotiat(?:e|es|ed|ion|ing)\b': 2,
}

PRESSURE_PATTERNS = {
    r'\braise(?:d|s|ing)?\b': 2,
    r'\bincrease(?:d|s|ing)?\b': 2,
    r'\bimpos(?:e|ed|es|ing)\b': 2,
    r'\bcharge(?:d|s|ing)?\b': 2,
    r'\btax(?:ed|es|ing)?\b': 2,
    r'\btariffs?\b': 1,
    r'\bsanctions?\b': 2,
    r'\bterminate(?:d|s|ing)?\b': 3,
    r'\bcancel(?:led|ed|s|ing)?\b': 2,
    r'\bunless\b': 2,
    r'\bwill not\b': 2,
    r'\beffective\b': 1,
    r'\bmust\b': 1,
    r'\bno deal\b': 2,
}

NEGATION_PATTERNS = [
    r'\bbut\b',
    r'\bwill not\b',
    r'\bunless\b',
    r'\bif not\b',
    r'\bno deal\b',
    r'\bnot enough\b',
]

SYSTEM_PROMPT = """You are labeling Trump policy-pressure events for a TACO project.
Return json only.

Goal:
Judge whether the text shows a clear retreat or softening within 48 hours after the anchor event.

Core decision rule:
- label_retreat = 1 only when the follow-up text clearly softens, pauses, delays, suspends, narrows, exempts, renegotiates away from, or otherwise backs off the anchor pressure.
- label_retreat = 0 when the follow-up keeps the pressure in place, escalates it, repeats the threat, or is too vague to show a real pullback.

Important distinctions:
- Mere negotiation, talking, meetings, optimism, or positive tone is NOT enough by itself.
- A deal only counts as retreat if the text clearly indicates the earlier threat is paused, softened, delayed, reduced, cancelled, replaced, or no longer being pursued in the same way.
- If the follow-up mixes softening with renewed threats or conditions, be conservative. If the retreat signal is not clearly dominant, output label_retreat = 0 and review_flag = 'yes'.
- Compare anchor and follow-up content. Do not judge from the last sentence alone.
- Use only the supplied text. Do not use outside knowledge.

Evidence rule:
- evidence_span must be a short direct quote from the provided text supporting your decision.
- If there is no clear evidence span, set label_retreat = 0, use low confidence, and set review_flag = 'yes'.

Output requirements:
- Output must be valid json.
- Required fields: label_retreat, confidence, evidence_span, review_flag, review_reason.
- label_retreat must be 0 or 1.
- confidence must be between 0 and 1.
- review_flag must be 'yes' or 'no'.
- review_reason may be a pipe-separated string such as low_conf|missing_evidence|rule_conflict|negation_hit.
""".strip()


def normalize_text(value: str) -> str:
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_follow_up_text(source_text: str) -> str:
    if pd.isna(source_text):
        return ''
    text = str(source_text)
    marker = '[Follow-up within 48h]'
    if marker not in text:
        return ''
    return text.split(marker, 1)[1].strip()


def find_hits(text: str, patterns: dict[str, int]) -> list[str]:
    hits = []
    for pattern in patterns:
        if re.search(pattern, text):
            hits.append(pattern)
    return hits


def unique_preserve(items: list[str]) -> list[str]:
    seen = set()
    output = []
    for item in items:
        if item and item not in seen:
            seen.add(item)
            output.append(item)
    return output


def build_rule_row(row: pd.Series) -> pd.Series:
    # 这里只对 follow-up 段落做规则扫描，因为我们想判断的是 anchor 后 48h 内是否出现软化。
    follow_up_text = extract_follow_up_text(row.get('source_text', ''))
    normalized_follow_up = normalize_text(follow_up_text)

    retreat_hits = find_hits(normalized_follow_up, RETREAT_PATTERNS)
    pressure_hits = find_hits(normalized_follow_up, PRESSURE_PATTERNS)
    negation_hits = [pattern for pattern in NEGATION_PATTERNS if re.search(pattern, normalized_follow_up)]

    retreat_score = sum(RETREAT_PATTERNS[pattern] for pattern in retreat_hits)
    pressure_score = sum(PRESSURE_PATTERNS[pattern] for pattern in pressure_hits)
    score_gap = retreat_score - pressure_score

    # 规则很保守：必须 retreat 侧得分明显更高，才直接给 rule_label=1。
    rule_label = int(retreat_score >= 3 and score_gap > 0)

    review_reasons = []
    if str(row.get('review_flag', '')).strip().lower() == 'yes' or not follow_up_text:
        review_reasons.append('missing_evidence')
    if negation_hits:
        review_reasons.append('negation_hit')
    if retreat_score == 0 and pressure_score == 0:
        review_reasons.append('low_conf')
    if retreat_hits and pressure_hits:
        review_reasons.append('rule_conflict')

    if score_gap >= 3:
        confidence_bucket = 'high'
    elif score_gap >= 1:
        confidence_bucket = 'medium'
    else:
        confidence_bucket = 'low'

    return pd.Series({
        'follow_up_text': follow_up_text,
        'rule_label': rule_label,
        'rule_confidence_bucket': confidence_bucket,
        'rule_retreat_score': retreat_score,
        'rule_pressure_score': pressure_score,
        'rule_score_gap': score_gap,
        'rule_retreat_hits': '|'.join(retreat_hits),
        'rule_pressure_hits': '|'.join(pressure_hits),
        'rule_negation_hits': '|'.join(negation_hits),
        'rule_review_flag': 'yes' if review_reasons else 'no',
        'rule_review_reason': '|'.join(unique_preserve(review_reasons)),
    })


rule_features = candidate_df.apply(build_rule_row, axis=1)
rule_df = pd.concat([candidate_df.copy(), rule_features], axis=1)
rule_df.to_csv(RULE_OUTPUT_PATH, index=False)

print(f'Wrote rule pre-tag file: {RULE_OUTPUT_PATH}')
rule_df[['rule_label', 'rule_confidence_bucket', 'rule_review_flag']].value_counts().head(10)


Wrote rule pre-tag file: /Users/horace/Coding/ML finance/project/data/processed/standardized/03_candidate_events_rule_pretag_v1.csv


rule_label  rule_confidence_bucket  rule_review_flag
0           low                     yes                 1275
                                    no                   202
1           high                    yes                  100
0           medium                  no                    95
1           medium                  yes                   94
0           medium                  yes                   85
1           high                    no                    35
Name: count, dtype: int64

## 2. Event Matching

这里统一整理事件匹配结果，并输出一个可直接回填到标签表的匹配表。


In [3]:
override_rank_map = {}
override_reason_map = {}
if not overrides_df.empty:
    override_rank_map = dict(zip(overrides_df['key_event_id'], overrides_df['preferred_match_rank']))
    override_reason_map = dict(zip(overrides_df['key_event_id'], overrides_df['override_reason']))

canonical_lookup = canonical_df.set_index('key_event_id').to_dict(orient='index')
match_candidates_sorted = match_candidates_df.sort_values(['key_event_id', 'matched_rank'])
candidate_lookup = candidate_df[[
    'event_id',
    'term_id',
    'source',
    'event_time_utc',
    'feature_anchor_date',
    'target_trade_date',
    'candidate_priority',
    'rule_text',
]].drop_duplicates('event_id')

selected_rows = []
for audit_row in audit_df.to_dict(orient='records'):
    key_event_id = audit_row['key_event_id']
    selected = {}
    selection_source = ''
    selection_reason = ''
    selected_rank = pd.NA

    if key_event_id in canonical_lookup:
        # core_top10 已经有人工确认过的 canonical match，优先信它。
        selected = canonical_lookup[key_event_id].copy()
        selection_source = 'canonical_core_top10'
        selection_reason = selected.get('selection_reason', 'canonical_core_top10')
        selected_rank = selected.get('selected_rank', selected.get('matched_rank', pd.NA))
    else:
        # 非 canonical 的条目回退到 match_candidates_v6；若存在 override，就用 override 指定的 rank。
        group = match_candidates_sorted.loc[match_candidates_sorted['key_event_id'] == key_event_id].copy()
        if not group.empty:
            preferred_rank = override_rank_map.get(key_event_id, 1)
            picked = group.loc[group['matched_rank'] == preferred_rank]
            if picked.empty:
                picked = group.head(1)
            selected = picked.iloc[0].to_dict()
            selection_source = 'override' if key_event_id in override_rank_map else 'default_top1'
            selection_reason = override_reason_map.get(key_event_id, 'default_top1')
            selected_rank = preferred_rank if key_event_id in override_rank_map else selected.get('matched_rank', pd.NA)

    merged = audit_row.copy()
    merged.update({
        'event_id': selected.get('event_id', pd.NA),
        'match_event_time': selected.get('match_event_time', pd.NA),
        'matched_rank': selected.get('matched_rank', pd.NA),
        'selected_rank': selected_rank,
        'selection_source': selection_source,
        'selection_reason': selection_reason,
        'keyword_overlap_terms': selected.get('keyword_overlap_terms', pd.NA),
        'rule_text_match': selected.get('rule_text', pd.NA),
    })
    selected_rows.append(merged)

audit_match_df = pd.DataFrame(selected_rows).merge(candidate_lookup, on='event_id', how='left', suffixes=('', '_candidate'))
audit_match_df['candidate_found'] = audit_match_df['event_id'].notna()
audit_match_df.to_csv(AUDIT_OUTPUT_PATH, index=False)

coverage_df = (
    audit_match_df
    .assign(match_status=lambda df: df['candidate_found'].map({True: 'matched', False: 'unmatched'}))
    .groupby(['audit_tier', 'match_status'], dropna=False)
    .size()
    .reset_index(name='n_events')
)
coverage_df.to_csv(AUDIT_COVERAGE_PATH, index=False)

print(f'Wrote audit match file: {AUDIT_OUTPUT_PATH}')
print(f'Wrote audit coverage file: {AUDIT_COVERAGE_PATH}')
audit_match_df[['key_event_id', 'audit_tier', 'event_name', 'event_id', 'selection_source', 'selection_reason']]


Wrote audit match file: /Users/horace/Coding/ML finance/project/data/manual/key_event_audit_matches_v1.csv
Wrote audit coverage file: /Users/horace/Coding/ML finance/project/data/manual/key_event_audit_coverage_v1.csv


,key_event_id,audit_tier,event_name,event_id,selection_source,selection_reason
0,K01,core_top10,NAFTA termination threat shifts to renegotiation,FIRST_TERM_TWITTER_857552956836786177,canonical_core_top10,default_top1
1,K02,extended_backup,Steel and aluminum tariff escalation with exem...,FIRST_TERM_TWITTER_972585290857672704,default_top1,default_top1
2,K03,core_top10,ZTE export ban softening,FIRST_TERM_TWITTER_996119678551552000,canonical_core_top10,default_top1
3,K04,extended_backup,Global baseline tariff scaled down in implemen...,NaN,default_top1,default_top1
4,K05,core_top10,EU auto tariff threat paused after Juncker mee...,FIRST_TERM_TWITTER_1022265839104483329,canonical_core_top10,default_top1
5,K06,core_top10,G20 China tariff increase delayed by 90-day truce,FIRST_TERM_TWITTER_1070110615627333632,canonical_core_top10,default_top1
6,K07,core_top10,Mexico tariff threat suspended after border deal,FIRST_TERM_TWITTER_1138023190020730880,canonical_core_top10,Rank 3 is the cleaner anchor for the Mexico ta...
7,K08,extended_backup,China tariff delay on electronics before holid...,FIRST_TERM_TWITTER_1163169730477395968,default_top1,default_top1
8,K09,core_top10,Phase One deal cancels December tariff round,FIRST_TERM_TWITTER_1205509125788098560,canonical_core_top10,default_top1
9,K10,extended_backup,France digital tax retaliation paused,FIRST_TERM_TWITTER_1217827468230434818,default_top1,default_top1


## 3. LLM Labeling Input Prep

这里不直接调用模型，而是先导出两份文件：

- `llm_labeling_batch_input_v1.jsonl`: 可送批处理或外部脚本
- `llm_labeling_response_template_v1.csv`: 你最终要回填的模板

`dispatch_priority` 的含义：

- `1`: 最高优先级样本
- `2`: 规则侧已经提示高风险，需要优先跑
- `3`: 高优先级候选
- `4`: 其他候选


In [4]:
label_input_df = rule_df.merge(
    audit_match_df[['event_id', 'key_event_id', 'audit_tier', 'expected_direction']],
    on='event_id',
    how='left',
)

label_input_df['dispatch_priority'] = 4
label_input_df.loc[label_input_df['candidate_priority'].eq('high'), 'dispatch_priority'] = 3
label_input_df.loc[label_input_df['rule_review_flag'].eq('yes'), 'dispatch_priority'] = 2
label_input_df.loc[label_input_df['audit_tier'].eq('core_top10'), 'dispatch_priority'] = 1

label_input_df = label_input_df.sort_values([
    'dispatch_priority',
    'term_id',
    'source',
    'event_time_utc',
]).reset_index(drop=True)

label_input_df.to_csv(LLM_INPUT_CSV_PATH, index=False)

batch_records = []
for row in label_input_df.to_dict(orient='records'):
    user_prompt = f"""Event metadata:
- event_id: {row['event_id']}
- source: {row['source']}
- term_id: {row['term_id']}
- event_time_utc: {row['event_time_utc']}
- rule_label: {row['rule_label']}
- key_event_id: {row.get('key_event_id', '')}
- audit_tier: {row.get('audit_tier', '')}

Task:
Decide whether this event shows a clear retreat / softening within 48 hours after the anchor event.

Source text:
{row['source_text']}

Return JSON only."""
    batch_records.append({
        'event_id': row['event_id'],
        'key_event_id': row.get('key_event_id', ''),
        'audit_tier': row.get('audit_tier', ''),
        'system_prompt': SYSTEM_PROMPT,
        'user_prompt': user_prompt,
    })

with LLM_BATCH_JSONL_PATH.open('w', encoding='utf-8') as f:
    for record in batch_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

response_template_df = label_input_df[[
    'event_id',
    'key_event_id',
    'audit_tier',
    'rule_label',
]].copy()
response_template_df['label_retreat'] = pd.NA
response_template_df['confidence'] = pd.NA
response_template_df['evidence_span'] = ''
response_template_df['review_flag'] = ''
response_template_df['review_reason'] = ''
response_template_df['model_name'] = ''
response_template_df['raw_response'] = ''
response_template_df.to_csv(LLM_TEMPLATE_PATH, index=False)

print(f'Wrote labeling input CSV: {LLM_INPUT_CSV_PATH}')
print(f'Wrote JSONL batch input: {LLM_BATCH_JSONL_PATH}')
print(f'Wrote response template CSV: {LLM_TEMPLATE_PATH}')
label_input_df[['dispatch_priority', 'event_id', 'key_event_id', 'audit_tier', 'rule_label']].head(10)


Wrote labeling input CSV: /Users/horace/Coding/ML finance/project/data/processed/standardized/03_candidate_events_labeling_input_v1.csv
Wrote JSONL batch input: /Users/horace/Coding/ML finance/project/data/manual/llm_labeling_batch_input_v1.jsonl
Wrote response template CSV: /Users/horace/Coding/ML finance/project/data/manual/llm_labeling_response_template_v1.csv


,dispatch_priority,event_id,key_event_id,audit_tier,rule_label
0,1,FIRST_TERM_TWITTER_857552956836786177,K01,core_top10,0
1,1,FIRST_TERM_TWITTER_996119678551552000,K03,core_top10,1
2,1,FIRST_TERM_TWITTER_1022265839104483329,K05,core_top10,0
3,1,FIRST_TERM_TWITTER_1070110615627333632,K06,core_top10,1
4,1,FIRST_TERM_TWITTER_1138023190020730880,K07,core_top10,1
5,1,FIRST_TERM_TWITTER_1205509125788098560,K09,core_top10,0
6,1,SECOND_TERM_TRUTHSOCIAL_113940711907400754,K13,core_top10,1
7,1,SECOND_TERM_TRUTHSOCIAL_113942189236610107,K14,core_top10,0
8,1,SECOND_TERM_TRUTHSOCIAL_115267382531822964,K12,core_top10,0
9,1,SECOND_TERM_TRUTHSOCIAL_115351840469973590,K15,core_top10,0


## 4. DeepSeek API Configuration and Runtime

这一部分负责配置 DeepSeek 调用方式，并将模型返回结果回填到 `llm_labeling_response_template_v1.csv`。

建议流程：先完成环境变量配置和 pilot batch 验证，再进行分批全量回填，最后运行合并单元生成正式标签表与 QC 表。

DeepSeek 当前官方文档要点：

- `base_url` 用 `https://api.deepseek.com`
- 模型可以用 `deepseek-chat`
- 结构化输出用 `response_format={"type": "json_object"}`
- prompt 里必须明确写 `json`
- 偶尔会返回空内容，所以代码里最好加重试

来源：
- https://api-docs.deepseek.com/
- https://api-docs.deepseek.com/api/create-chat-completion/
- https://api-docs.deepseek.com/guides/json_mode/


In [10]:
import os

# 在终端中提前设置 DEEPSEEK_API_KEY，例如：
# export DEEPSEEK_API_KEY="your_key_here"
os.environ.get("DEEPSEEK_API_KEY")

In [11]:
# 这一步只做配置检查，不会真的发 API 请求。
# 如果你还没有安装 openai SDK，请先执行：
# %pip install openai

DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
SAVE_EVERY = 10          # 每处理多少条就保存一次，避免中途中断丢结果
SLEEP_SECONDS = 0.8      # 两次请求之间的暂停，保守一点更稳
MAX_RETRIES = 2          # DeepSeek JSON 模式偶尔会空返回，所以允许少量重试
PILOT_SIZE = 20          # 第一次试跑建议不要超过 20 条
RANDOM_SEED = 42

api_key = os.environ.get('DEEPSEEK_API_KEY')
if api_key:
    print('DEEPSEEK_API_KEY is set. You can run the pilot batch next.')
else:
    print('DEEPSEEK_API_KEY is missing. In your terminal, run:')
    print('export DEEPSEEK_API_KEY="your_key_here"')


DEEPSEEK_API_KEY is set. You can run the pilot batch next.


In [14]:
# 这个单元包含和 DeepSeek 通信的核心函数。
# 设计思路：
# 1. 强制 json 输出，方便稳定解析
# 2. 做最小字段校验，避免坏结果直接写进模板
# 3. 自动保存，支持断点续跑
# 4. 对空结果和解析失败做重试

from openai import OpenAI


def load_labeling_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    """读取待标注主表和 LLM 回填模板，并显式修正模板列类型。"""
    label_input = pd.read_csv(LLM_INPUT_CSV_PATH)
    template = pd.read_csv(LLM_TEMPLATE_PATH)

    # 这些列后面会写入字符串。如果模板初始值全为空，pandas 很容易把它们误判成 float64，
    # 随后再写入 evidence_span 之类的文本时就会报 dtype 错误。
    text_cols = [
        'event_id',
        'key_event_id',
        'audit_tier',
        'evidence_span',
        'review_flag',
        'review_reason',
        'model_name',
        'raw_response',
    ]
    for col in text_cols:
        if col in template.columns:
            template[col] = template[col].astype('string')

    # 数值列也显式转型，避免后面比较、统计时出现 object / float 混杂。
    if 'rule_label' in template.columns:
        template['rule_label'] = pd.to_numeric(template['rule_label'], errors='coerce').astype('Int64')
    if 'label_retreat' in template.columns:
        template['label_retreat'] = pd.to_numeric(template['label_retreat'], errors='coerce')
    if 'confidence' in template.columns:
        template['confidence'] = pd.to_numeric(template['confidence'], errors='coerce')

    return label_input, template


def build_deepseek_client() -> OpenAI:
    """构造 DeepSeek 客户端。这里沿用 OpenAI SDK，但把 base_url 指向 DeepSeek。"""
    api_key = os.environ.get('DEEPSEEK_API_KEY')
    if not api_key:
        raise RuntimeError('DEEPSEEK_API_KEY is not set.')
    return OpenAI(api_key=api_key, base_url=DEEPSEEK_BASE_URL)


def build_system_prompt() -> str:
    # 注意：DeepSeek JSON Output 模式要求 prompt 中明确出现 json 字样。
    # 这里把定义写得更保守，尽量减少把“谈判/表态改善”误判成 retreat。
    return """You are labeling Trump policy-pressure events for a TACO project.
Return json only.

Task:
Decide whether the text shows a clear retreat or softening within 48 hours after the anchor event.

Decision standard:
- label_retreat = 1 only if the follow-up clearly backs off the anchor pressure.
- Examples that may support label_retreat = 1: pause, delay, suspend, exemption, waiver, cancel a planned escalation, narrow the scope, switch from termination to renegotiation, or otherwise clearly soften the original threat.
- label_retreat = 0 if the follow-up continues the threat, escalates it, repeats it, or is too vague to show a real pullback.

Important distinctions:
- A meeting, call, negotiation, optimistic tone, or talk of a possible deal is NOT enough by itself.
- A deal counts only if the text clearly indicates that the earlier pressure was paused, softened, delayed, reduced, cancelled, or replaced by a less aggressive stance.
- If the follow-up contains both softening and renewed threats, be conservative. If the retreat signal is not clearly dominant, use label_retreat = 0 and review_flag = 'yes'.
- Compare anchor and follow-up content rather than reading the final sentence in isolation.
- Use only the supplied text. Do not use outside knowledge.

Evidence rule:
- evidence_span must be a short direct quote copied from the provided text.
- If there is no clear supporting quote, use label_retreat = 0, low confidence, and review_flag = 'yes'.

Output requirements:
- Output must be valid json.
- Required fields: label_retreat, confidence, evidence_span, review_flag, review_reason.
- label_retreat must be 0 or 1.
- confidence must be between 0 and 1.
- review_flag must be 'yes' or 'no'.
- review_reason may be a pipe-separated string such as low_conf|missing_evidence|rule_conflict|negation_hit.

Example json:
{
  "label_retreat": 1,
  "confidence": 0.82,
  "evidence_span": "we have agreed to pause the tariffs",
  "review_flag": "no",
  "review_reason": ""
}
""".strip()


def build_user_prompt(row: pd.Series) -> str:
    # 元信息保留在 prompt 里，方便后面人工追踪，也让模型知道该事件是否属于 core_top10。
    # 这里额外加入 decision checklist，帮助模型按固定顺序判断。
    return f"""json only

event_id: {row['event_id']}
source: {row['source']}
term_id: {row['term_id']}
event_time_utc: {row['event_time_utc']}
rule_label: {row['rule_label']}
rule_review_reason: {row.get('rule_review_reason', '')}
key_event_id: {row.get('key_event_id', '')}
audit_tier: {row.get('audit_tier', '')}
expected_direction: {row.get('expected_direction', '')}

Decision checklist:
1. What is the anchor pressure or threat?
2. Within 48h, is there a clear textual sign that this pressure was softened, paused, delayed, narrowed, exempted, renegotiated away from, or otherwise backed off?
3. If there is mixed evidence, is the retreat signal clearly stronger than the renewed threat signal?
4. Can you quote a short direct evidence_span from the text?
5. If any of the answers above are uncertain, return label_retreat = 0 with lower confidence and review_flag = 'yes'.

Source text:
{row['source_text']}
""".strip()


def strip_code_fences(text: str) -> str:
    if not text:
        return text
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    return text.strip()


def parse_llm_json(raw_content: str) -> dict[str, Any]:
    """解析模型返回，并做最小字段规范化。"""
    content = strip_code_fences(raw_content)
    if not content:
        raise ValueError('Empty content returned by model.')

    parsed = json.loads(content)

    if 'label_retreat' not in parsed:
        raise ValueError('Missing label_retreat.')
    if 'confidence' not in parsed:
        raise ValueError('Missing confidence.')

    label_retreat = int(parsed['label_retreat'])
    if label_retreat not in (0, 1):
        raise ValueError(f'Invalid label_retreat: {label_retreat}')

    confidence = float(parsed['confidence'])
    confidence = max(0.0, min(1.0, confidence))

    evidence_span = str(parsed.get('evidence_span', '') or '')
    review_flag = str(parsed.get('review_flag', 'yes') or 'yes').strip().lower()
    if review_flag not in {'yes', 'no'}:
        review_flag = 'yes'

    review_reason = str(parsed.get('review_reason', '') or '')

    return {
        'label_retreat': label_retreat,
        'confidence': confidence,
        'evidence_span': evidence_span,
        'review_flag': review_flag,
        'review_reason': review_reason,
        'raw_response': content,
    }


def call_deepseek_label(client: OpenAI, row: pd.Series, max_retries: int = MAX_RETRIES) -> dict[str, Any]:
    """调用 DeepSeek 给单条事件打标签。"""
    system_prompt = build_system_prompt()
    user_prompt = build_user_prompt(row)

    last_error = None
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                temperature=0,
                max_tokens=300,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt},
                ],
            )
            raw_content = response.choices[0].message.content
            parsed = parse_llm_json(raw_content)
            parsed['model_name'] = DEEPSEEK_MODEL
            return parsed
        except Exception as exc:
            last_error = exc
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
            else:
                raise RuntimeError(f'Failed after retries for event_id={row["event_id"]}: {exc}') from exc

    raise RuntimeError(f'Unexpected failure: {last_error}')


def get_unlabeled_rows(label_input_df: pd.DataFrame, template_df: pd.DataFrame) -> pd.DataFrame:
    """只返回还没有填 label_retreat 的行，方便断点续跑。"""
    pending_ids = set(template_df.loc[template_df['label_retreat'].isna(), 'event_id'])
    return label_input_df.loc[label_input_df['event_id'].isin(pending_ids)].copy()


def save_template(template_df: pd.DataFrame) -> None:
    template_df.to_csv(LLM_TEMPLATE_PATH, index=False)


def write_result_to_template(template_df: pd.DataFrame, event_id: str, result: dict[str, Any]) -> None:
    """把单条 LLM 结果写回模板。"""
    idx = template_df.index[template_df['event_id'] == event_id][0]

    # 再做一层保险：如果文本列被错误推断成数值型，先转成 string，再写入文本结果。
    text_fields = ['evidence_span', 'review_flag', 'review_reason', 'model_name', 'raw_response']
    for field in text_fields:
        if field in template_df.columns and str(template_df[field].dtype) not in {'string', 'object'}:
            template_df[field] = template_df[field].astype('string')

    for key, value in result.items():
        template_df.at[idx, key] = value


def run_labeling_batch(rows_to_run: pd.DataFrame, sleep_seconds: float = SLEEP_SECONDS, save_every: int = SAVE_EVERY) -> pd.DataFrame:
    """按批次运行 DeepSeek 标注，并持续保存模板。"""
    if rows_to_run.empty:
        print('No rows to label in this batch.')
        return pd.read_csv(LLM_TEMPLATE_PATH)

    client = build_deepseek_client()
    _, template_df = load_labeling_frames()

    completed = 0
    failed_event_ids = []

    for _, row in rows_to_run.iterrows():
        event_id = row['event_id']
        try:
            result = call_deepseek_label(client, row)
            write_result_to_template(template_df, event_id, result)
            completed += 1
        except Exception as exc:
            failed_event_ids.append(event_id)
            print(f'[FAILED] {event_id}: {exc}')

        if completed > 0 and completed % save_every == 0:
            save_template(template_df)
            print(f'Saved progress after {completed} completed rows.')

        time.sleep(sleep_seconds)

    save_template(template_df)
    print(f'Batch finished. completed={completed}, failed={len(failed_event_ids)}')
    if failed_event_ids:
        print('Failed event_ids:', failed_event_ids[:20])
    return template_df


## 5. Pilot Batch

建议先运行小批量样本，验证返回格式、证据片段和标签质量，再进入全量处理。

可优先选择：

- 最高优先级样本
- `dispatch_priority <= 2`
- 或者直接前 20 条

完成后建议抽样检查模板中的若干记录，重点确认：

- `evidence_span` 不是胡乱截的
- `confidence` 分布看起来合理
- `review_flag` 没有全部变成同一个值


In [13]:
label_input_df, template_df = load_labeling_frames()
pending_df = get_unlabeled_rows(label_input_df, template_df)

# 优先用 core_top10 + 高风险样本做小试跑。
pilot_df = (
    pending_df
    .sort_values(['dispatch_priority', 'event_time_utc'])
    .head(PILOT_SIZE)
    .copy()
)

print(f'pending rows: {len(pending_df):,}')
print(f'pilot rows: {len(pilot_df):,}')
pilot_df[['dispatch_priority', 'event_id', 'key_event_id', 'audit_tier', 'rule_label', 'rule_review_reason']].head(20)


pending rows: 1,886
pilot rows: 20


,dispatch_priority,event_id,key_event_id,audit_tier,rule_label,rule_review_reason
0,1,FIRST_TERM_TWITTER_857552956836786177,K01,core_top10,0,rule_conflict
1,1,FIRST_TERM_TWITTER_996119678551552000,K03,core_top10,1,NaN
2,1,FIRST_TERM_TWITTER_1022265839104483329,K05,core_top10,0,low_conf
3,1,FIRST_TERM_TWITTER_1070110615627333632,K06,core_top10,1,negation_hit|rule_conflict
4,1,FIRST_TERM_TWITTER_1138023190020730880,K07,core_top10,1,negation_hit|rule_conflict
5,1,FIRST_TERM_TWITTER_1205509125788098560,K09,core_top10,0,negation_hit|rule_conflict
6,1,SECOND_TERM_TRUTHSOCIAL_113940711907400754,K13,core_top10,1,negation_hit|rule_conflict
7,1,SECOND_TERM_TRUTHSOCIAL_113942189236610107,K14,core_top10,0,negation_hit|low_conf
8,1,SECOND_TERM_TRUTHSOCIAL_115267382531822964,K12,core_top10,0,negation_hit
9,1,SECOND_TERM_TRUTHSOCIAL_115351840469973590,K15,core_top10,0,low_conf


In [14]:
# 真正发请求的单元。
# 建议第一次只跑 pilot_df，不要直接跑全量。

pilot_result_df = run_labeling_batch(pilot_df, sleep_seconds=SLEEP_SECONDS, save_every=5)
pilot_result_df.loc[pilot_result_df['event_id'].isin(pilot_df['event_id']), [
    'event_id', 'label_retreat', 'confidence', 'evidence_span', 'review_flag', 'review_reason', 'model_name'
]].head(20)


Saved progress after 5 completed rows.
Saved progress after 10 completed rows.
Saved progress after 15 completed rows.
Saved progress after 20 completed rows.
Batch finished. completed=20, failed=0


,event_id,label_retreat,confidence,evidence_span,review_flag,review_reason,model_name
0,FIRST_TERM_TWITTER_857552956836786177,0.0,0.75,subject to the fact that if we do not reach a ...,yes,rule_conflict|low_conf,deepseek-chat
1,FIRST_TERM_TWITTER_996119678551552000,0.0,0.75,Nothing has happened with ZTE except as it per...,yes,low_conf|missing_evidence,deepseek-chat
2,FIRST_TERM_TWITTER_1022265839104483329,0.0,0.60,,yes,low_conf|missing_evidence,deepseek-chat
3,FIRST_TERM_TWITTER_1070110615627333632,0.0,0.60,,yes,low_conf|missing_evidence,deepseek-chat
4,FIRST_TERM_TWITTER_1138023190020730880,0.0,0.85,if for any reason the approval is not forthcom...,no,,deepseek-chat
5,FIRST_TERM_TWITTER_1205509125788098560,1.0,0.95,The Penalty Tariffs set for December 15th will...,no,,deepseek-chat
6,SECOND_TERM_TRUTHSOCIAL_113940711907400754,1.0,0.95,the Tariffs announced on Saturday will be paus...,no,,deepseek-chat
7,SECOND_TERM_TRUTHSOCIAL_113942189236610107,1.0,0.95,the Tariffs announced on Saturday will be paus...,no,,deepseek-chat
8,SECOND_TERM_TRUTHSOCIAL_115267382531822964,0.0,0.95,,no,,deepseek-chat
9,SECOND_TERM_TRUTHSOCIAL_115351840469973590,0.0,0.60,"Don't worry about China, it will all be fine!",yes,low_conf|missing_evidence,deepseek-chat


## 6. Pilot Review

这一部分用于对 pilot 结果做有针对性的抽样复核。

优先看：

- 最高优先级样本
- `confidence < 0.6`
- `evidence_span` 为空
- `rule_label` 和模型判断明显冲突的样本


In [16]:
# 这里不要重复 merge key_event_id / audit_tier / rule_label，
# 因为这些列在 LLM template 里本来就有，再 merge 一次会被 pandas 自动加后缀，
# 后面按原列名取值时就会 KeyError。

pilot_check_df = pd.read_csv(LLM_TEMPLATE_PATH).merge(
    label_input_df[['event_id', 'source_text', 'dispatch_priority']],
    on='event_id',
    how='left',
)

pilot_subset = pilot_check_df.loc[
    pilot_check_df['event_id'].isin(pilot_df['event_id'])
].copy()

pilot_subset = pilot_subset.sort_values(
    ['dispatch_priority', 'confidence', 'event_id'],
    na_position='last'
)

pilot_subset[[
    'event_id',
    'key_event_id',
    'audit_tier',
    'rule_label',
    'label_retreat',
    'confidence',
    'review_flag',
    'review_reason',
    'evidence_span',
]].head(20)


,event_id,key_event_id,audit_tier,rule_label,label_retreat,confidence,review_flag,review_reason,evidence_span
2,FIRST_TERM_TWITTER_1022265839104483329,K05,core_top10,0,0.0,0.60,yes,low_conf|missing_evidence,NaN
3,FIRST_TERM_TWITTER_1070110615627333632,K06,core_top10,1,0.0,0.60,yes,low_conf|missing_evidence,NaN
9,SECOND_TERM_TRUTHSOCIAL_115351840469973590,K15,core_top10,0,0.0,0.60,yes,low_conf|missing_evidence,"Don't worry about China, it will all be fine!"
0,FIRST_TERM_TWITTER_857552956836786177,K01,core_top10,0,0.0,0.75,yes,rule_conflict|low_conf,subject to the fact that if we do not reach a ...
1,FIRST_TERM_TWITTER_996119678551552000,K03,core_top10,1,0.0,0.75,yes,low_conf|missing_evidence,Nothing has happened with ZTE except as it per...
4,FIRST_TERM_TWITTER_1138023190020730880,K07,core_top10,1,0.0,0.85,no,NaN,if for any reason the approval is not forthcom...
5,FIRST_TERM_TWITTER_1205509125788098560,K09,core_top10,0,1.0,0.95,no,NaN,The Penalty Tariffs set for December 15th will...
6,SECOND_TERM_TRUTHSOCIAL_113940711907400754,K13,core_top10,1,1.0,0.95,no,NaN,the Tariffs announced on Saturday will be paus...
7,SECOND_TERM_TRUTHSOCIAL_113942189236610107,K14,core_top10,0,1.0,0.95,no,NaN,the Tariffs announced on Saturday will be paus...
8,SECOND_TERM_TRUTHSOCIAL_115267382531822964,K12,core_top10,0,0.0,0.95,no,NaN,NaN


In [23]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

manual_path = ROOT / "data" / "manual" / "core_top10_manual_review_v1.csv"
label_input_path = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_labeling_input_v1.csv"
output_path = ROOT / "data" / "manual" / "core_top10_manual_review_with_text_v1.csv"

manual_df = pd.read_csv(manual_path)
label_input_df = pd.read_csv(label_input_path)

manual_df.columns = manual_df.columns.str.strip()
label_input_df.columns = label_input_df.columns.str.strip()

manual_with_text_df = manual_df.merge(
    label_input_df[["event_id", "source_text"]],
    on="event_id",
    how="left",
)

def split_source_text(text: str) -> pd.Series:
    if pd.isna(text):
        return pd.Series({
            "anchor_text": "",
            "follow_up_text_48h": "",
        })

    text = str(text)

    anchor_part = ""
    follow_up_part = ""

    if "[Follow-up within 48h]" in text:
        parts = text.split("[Follow-up within 48h]", 1)
        anchor_part = parts[0]
        follow_up_part = parts[1]
    else:
        anchor_part = text
        follow_up_part = ""

    # 去掉标题标签，保留更干净的正文
    anchor_part = anchor_part.replace("[Anchor]", "").strip()
    follow_up_part = follow_up_part.strip()

    return pd.Series({
        "anchor_text": anchor_part,
        "follow_up_text_48h": follow_up_part,
    })

text_parts = manual_with_text_df["source_text"].apply(split_source_text)
manual_with_text_df = pd.concat([manual_with_text_df, text_parts], axis=1)

preferred_cols = [
    "key_event_id",
    "event_name",
    "expected_direction",
    "event_id",
    "rule_label",
    "llm_label_retreat",
    "llm_confidence",
    "llm_review_flag",
    "llm_review_reason",
    "llm_evidence_span",
    "manual_match_ok",
    "manual_label_retreat",
    "manual_evidence_ok",
    "manual_review_needed",
    "manual_review_reason",
    "manual_evidence_span",
    "manual_decision",
    "manual_note",
    "anchor_text",
    "follow_up_text_48h",
    "source_text",
]

existing_cols = [c for c in preferred_cols if c in manual_with_text_df.columns]
remaining_cols = [c for c in manual_with_text_df.columns if c not in existing_cols]
manual_with_text_df = manual_with_text_df[existing_cols + remaining_cols]

manual_with_text_df.to_csv(output_path, index=False)
print(f"Wrote: {output_path}")


Wrote: /Users/horace/Coding/ML finance/project/data/manual/core_top10_manual_review_with_text_v1.csv


In [24]:
from pathlib import Path
import shutil

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

template_path = ROOT / "data" / "manual" / "llm_labeling_response_template_v1.csv"
backup_path = ROOT / "data" / "manual" / "llm_labeling_response_template_v1_before_manual_top10_backup.csv"

shutil.copy2(template_path, backup_path)
print(f"Backed up current template to: {backup_path}")


Backed up current template to: /Users/horace/Coding/ML finance/project/data/manual/llm_labeling_response_template_v1_before_manual_top10_backup.csv


In [25]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

template_path = ROOT / "data" / "manual" / "llm_labeling_response_template_v1.csv"
manual_review_path = ROOT / "data" / "manual" / "core_top10_manual_review_with_text_v1.csv"

template_df = pd.read_csv(template_path)
manual_df = pd.read_csv(manual_review_path)

template_df.columns = template_df.columns.str.strip()
manual_df.columns = manual_df.columns.str.strip()

# 只处理你已经人工给出 manual_decision 的 core_top10
reviewed_df = manual_df.loc[manual_df["manual_decision"].notna()].copy()

valid_decisions = {"accept_llm", "override_llm", "needs_review"}
bad_decisions = set(reviewed_df["manual_decision"].dropna()) - valid_decisions
if bad_decisions:
    raise ValueError(f"Unexpected manual_decision values: {bad_decisions}")

# override_llm 必须给 manual_label_retreat
override_missing = reviewed_df.loc[
    reviewed_df["manual_decision"].eq("override_llm") &
    reviewed_df["manual_label_retreat"].isna()
]
if not override_missing.empty:
    raise ValueError(
        "Some override_llm rows are missing manual_label_retreat: "
        + ", ".join(override_missing["key_event_id"].astype(str).tolist())
    )

def merge_reason(existing: str, new_reason: str) -> str:
    parts = []
    for chunk in [existing, new_reason]:
        if pd.notna(chunk) and str(chunk).strip():
            parts.extend([x for x in str(chunk).split("|") if x])
    # 去重但保留顺序
    seen = set()
    out = []
    for x in parts:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return "|".join(out)

updated = 0

for _, row in reviewed_df.iterrows():
    event_id = row["event_id"]
    idx_list = template_df.index[template_df["event_id"] == event_id].tolist()
    if not idx_list:
        print(f"[WARN] event_id not found in template: {event_id}")
        continue

    idx = idx_list[0]
    decision = str(row["manual_decision"]).strip()

    if decision == "accept_llm":
        # 接受 LLM，不改主标签；如果你补了人工证据，就用人工证据覆盖
        if pd.notna(row.get("manual_evidence_span")) and str(row["manual_evidence_span"]).strip():
            template_df.at[idx, "evidence_span"] = str(row["manual_evidence_span"]).strip()

        # 记录这条已经经过 core_top10 人工终审
        template_df.at[idx, "review_reason"] = merge_reason(
            template_df.at[idx, "review_reason"] if "review_reason" in template_df.columns else "",
            "core_top10_manual_checked"
        )

    elif decision == "override_llm":
        template_df.at[idx, "label_retreat"] = int(row["manual_label_retreat"])
        template_df.at[idx, "review_flag"] = "yes"

        if pd.notna(row.get("manual_evidence_span")) and str(row["manual_evidence_span"]).strip():
            template_df.at[idx, "evidence_span"] = str(row["manual_evidence_span"]).strip()

        override_reason = "manual_override_top10"
        if pd.notna(row.get("manual_review_reason")) and str(row["manual_review_reason"]).strip():
            override_reason = merge_reason(override_reason, str(row["manual_review_reason"]).strip())

        template_df.at[idx, "review_reason"] = merge_reason(
            template_df.at[idx, "review_reason"] if "review_reason" in template_df.columns else "",
            override_reason
        )

        current_model = ""
        if "model_name" in template_df.columns and pd.notna(template_df.at[idx, "model_name"]):
            current_model = str(template_df.at[idx, "model_name"]).strip()
        template_df.at[idx, "model_name"] = (
            current_model + "|manual_top10" if current_model else "manual_top10"
        )

    elif decision == "needs_review":
        template_df.at[idx, "review_flag"] = "yes"
        template_df.at[idx, "review_reason"] = merge_reason(
            template_df.at[idx, "review_reason"] if "review_reason" in template_df.columns else "",
            "manual_top10_needs_review"
        )

    updated += 1

template_df.to_csv(template_path, index=False)
print(f"Applied manual review decisions for {updated} core_top10 rows.")
print(f"Updated template saved to: {template_path}")


Applied manual review decisions for 10 core_top10 rows.
Updated template saved to: /Users/horace/Coding/ML finance/project/data/manual/llm_labeling_response_template_v1.csv


## 7. Full Run

当 pilot 结果确认稳定后，再执行这一部分完成分批全量回填。

建议做法：

- 每次跑 `50` 到 `200` 条，而不是一次性全量
- 每跑完一批就保存并抽看几条
- 如果中途断掉，重新运行这个单元会自动从未完成的行继续


In [26]:
# 你可以手动调整这几个参数。
RUN_ONLY_PRIORITY_LE = None   # 例如设成 2，表示先只跑 dispatch_priority <= 2
MAX_ROWS_THIS_BATCH = 300     # 每一批最多跑多少条

label_input_df, template_df = load_labeling_frames()
pending_df = get_unlabeled_rows(label_input_df, template_df)

if RUN_ONLY_PRIORITY_LE is not None:
    pending_df = pending_df.loc[pending_df['dispatch_priority'] <= RUN_ONLY_PRIORITY_LE].copy()

full_batch_df = (
    pending_df
    .sort_values(['dispatch_priority', 'term_id', 'source', 'event_time_utc'])
    .head(MAX_ROWS_THIS_BATCH)
    .copy()
)

print(f'rows in this batch: {len(full_batch_df):,}')
full_batch_df[['dispatch_priority', 'event_id', 'key_event_id', 'audit_tier', 'rule_label']].head(20)


rows in this batch: 300


,dispatch_priority,event_id,key_event_id,audit_tier,rule_label
1586,3,FIRST_TERM_TWITTER_983774270886236161,NaN,NaN,0
1587,3,FIRST_TERM_TWITTER_984017894240604161,NaN,NaN,0
1588,3,FIRST_TERM_TWITTER_989144989690187776,NaN,NaN,0
1589,3,FIRST_TERM_TWITTER_993562242124865536,NaN,NaN,0
1590,3,FIRST_TERM_TWITTER_993813485745295360,NaN,NaN,0
1591,3,FIRST_TERM_TWITTER_1001404640796336128,NaN,NaN,0
1592,3,FIRST_TERM_TWITTER_1002539852171304960,NaN,NaN,0
1593,3,FIRST_TERM_TWITTER_1002971013313908738,NaN,NaN,0
1594,3,FIRST_TERM_TWITTER_1003617811254710273,NaN,NaN,0
1595,3,FIRST_TERM_TWITTER_1008333717545410560,NaN,NaN,0


In [27]:
# 真正执行当前批次。
# 每跑完一批，都建议先停下来做一次简单检查，再决定是否继续下一批。

batch_result_df = run_labeling_batch(full_batch_df, sleep_seconds=SLEEP_SECONDS, save_every=SAVE_EVERY)
remaining_after_batch = batch_result_df['label_retreat'].isna().sum()
print(f'Remaining unlabeled rows: {remaining_after_batch:,}')


Saved progress after 10 completed rows.
Saved progress after 20 completed rows.
Saved progress after 30 completed rows.
Saved progress after 40 completed rows.
Saved progress after 50 completed rows.
Saved progress after 60 completed rows.
Saved progress after 70 completed rows.
Saved progress after 80 completed rows.
Saved progress after 90 completed rows.
Saved progress after 100 completed rows.
Saved progress after 110 completed rows.
Saved progress after 120 completed rows.
Saved progress after 130 completed rows.
Saved progress after 140 completed rows.
Saved progress after 150 completed rows.
Saved progress after 160 completed rows.
Saved progress after 170 completed rows.
Saved progress after 180 completed rows.
Saved progress after 190 completed rows.
Saved progress after 200 completed rows.
Saved progress after 210 completed rows.
Saved progress after 220 completed rows.
Saved progress after 230 completed rows.
Saved progress after 240 completed rows.
Saved progress after 250 

## 8. Review Queue and Auto-Pass Sample

这一部分用于生成后续复核所需的重点样本集合：

- 把必须人工看的样本单独导出来
- 从自动通过样本中抽一个 10% 的检查样本

复核池规则：

- `confidence < 0.6`
- `review_flag == yes`
- `evidence_span` 为空
- 最高优先级样本
- `rule_label` 与 `label_retreat` 冲突


In [28]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

LLM_INPUT_CSV_PATH = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_labeling_input_v1.csv"
LLM_TEMPLATE_PATH = ROOT / "data" / "manual" / "llm_labeling_response_template_v1.csv"
REVIEW_QUEUE_PATH = ROOT / "data" / "manual" / "label_review_queue_v1.csv"
AUTO_PASS_SAMPLE_PATH = ROOT / "data" / "manual" / "label_auto_pass_sample_v1.csv"
RANDOM_SEED = 42

def load_labeling_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    label_input = pd.read_csv(LLM_INPUT_CSV_PATH)
    template = pd.read_csv(LLM_TEMPLATE_PATH)

    text_cols = [
        "event_id", "key_event_id", "audit_tier",
        "evidence_span", "review_flag", "review_reason",
        "model_name", "raw_response",
    ]
    for col in text_cols:
        if col in template.columns:
            template[col] = template[col].astype("string")

    if "rule_label" in template.columns:
        template["rule_label"] = pd.to_numeric(template["rule_label"], errors="coerce").astype("Int64")
    if "label_retreat" in template.columns:
        template["label_retreat"] = pd.to_numeric(template["label_retreat"], errors="coerce")
    if "confidence" in template.columns:
        template["confidence"] = pd.to_numeric(template["confidence"], errors="coerce")

    return label_input, template

print("Helper loaded. Safe to run section 8.")

Helper loaded. Safe to run section 8.


In [29]:
label_input_df, template_df = load_labeling_frames()

# 这里只从 template 取 LLM / 人工更新后的字段，避免和 label_input_df 里的同名列混淆
review_base_df = label_input_df.merge(
    template_df[[
        "event_id",
        "label_retreat",
        "confidence",
        "evidence_span",
        "review_flag",
        "review_reason",
        "model_name",
        "raw_response",
    ]],
    on="event_id",
    how="left",
    suffixes=("", "_llm"),
)

review_base_df["rule_llm_conflict"] = (
    review_base_df["label_retreat"].notna() &
    (
        pd.to_numeric(review_base_df["rule_label"], errors="coerce") !=
        pd.to_numeric(review_base_df["label_retreat"], errors="coerce")
    )
).astype(int)

# 用 template 里的 review_flag / review_reason 作为最终自动复核信号
review_queue_df = review_base_df.loc[
    (pd.to_numeric(review_base_df["confidence"], errors="coerce") < 0.6) |
    (review_base_df["review_flag"].fillna("").str.lower() == "yes") |
    (review_base_df["evidence_span"].fillna("").str.strip() == "") |
    (review_base_df["audit_tier"] == "core_top10") |
    (review_base_df["rule_llm_conflict"] == 1)
].copy()

review_queue_df = review_queue_df.sort_values(
    ["dispatch_priority", "audit_tier", "confidence", "event_time_utc"],
    na_position="last",
)

review_queue_df.to_csv(REVIEW_QUEUE_PATH, index=False)

# 自动通过样本抽 10%
auto_pass_df = review_base_df.loc[
    review_base_df["label_retreat"].notna() &
    (review_base_df["review_flag"].fillna("").str.lower() != "yes") &
    (review_base_df["evidence_span"].fillna("").str.strip() != "") &
    (review_base_df["rule_llm_conflict"] == 0) &
    (pd.to_numeric(review_base_df["confidence"], errors="coerce") >= 0.6)
].copy()

sample_n = max(1, int(round(len(auto_pass_df) * 0.10))) if len(auto_pass_df) else 0
if sample_n > 0:
    auto_pass_sample_df = auto_pass_df.sample(
        n=min(sample_n, len(auto_pass_df)),
        random_state=RANDOM_SEED,
    )
else:
    auto_pass_sample_df = auto_pass_df.head(0).copy()

auto_pass_sample_df.to_csv(AUTO_PASS_SAMPLE_PATH, index=False)

print(f"Wrote review queue: {REVIEW_QUEUE_PATH}")
print(f"Wrote auto-pass sample: {AUTO_PASS_SAMPLE_PATH}")
print(f"review queue rows: {len(review_queue_df):,}")
print(f"auto-pass sample rows: {len(auto_pass_sample_df):,}")

review_queue_df[[
    "event_id",
    "key_event_id",
    "audit_tier",
    "rule_label",
    "label_retreat",
    "confidence",
    "review_flag",
    "review_reason",
]].head(20)


Wrote review queue: /Users/horace/Coding/ML finance/project/data/manual/label_review_queue_v1.csv
Wrote auto-pass sample: /Users/horace/Coding/ML finance/project/data/manual/label_auto_pass_sample_v1.csv
review queue rows: 1,445
auto-pass sample rows: 44


,event_id,key_event_id,audit_tier,rule_label,label_retreat,confidence,review_flag,review_reason
2,FIRST_TERM_TWITTER_1022265839104483329,K05,core_top10,0,1.0,0.60,no,low_conf|missing_evidence|manual_override_top10
3,FIRST_TERM_TWITTER_1070110615627333632,K06,core_top10,1,0.0,0.60,no,low_conf|missing_evidence|core_top10_manual_ch...
9,SECOND_TERM_TRUTHSOCIAL_115351840469973590,K15,core_top10,0,0.0,0.60,no,low_conf|missing_evidence|core_top10_manual_ch...
0,FIRST_TERM_TWITTER_857552956836786177,K01,core_top10,0,0.0,0.75,no,rule_conflict|low_conf|core_top10_manual_checked
1,FIRST_TERM_TWITTER_996119678551552000,K03,core_top10,1,0.0,0.75,no,low_conf|missing_evidence|core_top10_manual_ch...
4,FIRST_TERM_TWITTER_1138023190020730880,K07,core_top10,1,0.0,0.85,no,core_top10_manual_checked
5,FIRST_TERM_TWITTER_1205509125788098560,K09,core_top10,0,1.0,0.95,no,core_top10_manual_checked
6,SECOND_TERM_TRUTHSOCIAL_113940711907400754,K13,core_top10,1,1.0,0.95,no,core_top10_manual_checked
7,SECOND_TERM_TRUTHSOCIAL_113942189236610107,K14,core_top10,0,1.0,0.95,no,core_top10_manual_checked
8,SECOND_TERM_TRUTHSOCIAL_115267382531822964,K12,core_top10,0,0.0,0.95,no,core_top10_manual_checked


## 9. Merge LLM Outputs Back

当模板中已经累计足够的有效标签后，再执行这一部分完成结果合并。

它会：

1. 把规则结果、LLM 结果和匹配字段合并起来
2. 自动生成 `rule_llm_conflict`
3. 汇总 `review_reason`
4. 输出正式标签表 `labels_mvp_v1.csv`
5. 输出质控表 `label_qc_audit_v1.csv`


In [31]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RULE_OUTPUT_PATH = ROOT / "data" / "processed" / "standardized" / "03_candidate_events_rule_pretag_v1.csv"
AUDIT_OUTPUT_PATH = ROOT / "data" / "manual" / "key_event_audit_matches_v1.csv"
LLM_TEMPLATE_PATH = ROOT / "data" / "manual" / "llm_labeling_response_template_v1.csv"
FINAL_LABEL_PATH = ROOT / "data" / "processed" / "standardized" / "labels_mvp_v1.csv"
QC_AUDIT_PATH = ROOT / "data" / "manual" / "label_qc_audit_v1.csv"

rule_df = pd.read_csv(RULE_OUTPUT_PATH)
audit_match_df = pd.read_csv(AUDIT_OUTPUT_PATH)
llm_response_df = pd.read_csv(LLM_TEMPLATE_PATH)

print("rule_df:", rule_df.shape)
print("audit_match_df:", audit_match_df.shape)
print("llm labeled rows:", llm_response_df["label_retreat"].notna().sum())

rule_df: (1886, 36)
audit_match_df: (15, 25)
llm labeled rows: 1886


In [30]:
llm_response_df = pd.read_csv(LLM_TEMPLATE_PATH)

if llm_response_df["label_retreat"].notna().sum() == 0:
    print("LLM response template exists, but no labels are filled yet.")
else:
    final_df = rule_df.merge(
        llm_response_df[[
            "event_id",
            "label_retreat",
            "confidence",
            "evidence_span",
            "review_flag",
            "review_reason",
            "model_name",
            "raw_response",
        ]],
        on="event_id",
        how="left",
    ).merge(
        audit_match_df[["event_id", "key_event_id", "audit_tier", "expected_direction"]],
        on="event_id",
        how="left",
    )

    final_df["label_retreat"] = pd.to_numeric(final_df["label_retreat"], errors="coerce").astype("Int64")
    final_df["confidence"] = pd.to_numeric(final_df["confidence"], errors="coerce")
    final_df["evidence_span"] = final_df["evidence_span"].fillna("")
    final_df["review_reason"] = final_df["review_reason"].fillna("")

    # 只有已经完成标注的样本才参与最终 review / QC
    final_df["is_labeled"] = final_df["label_retreat"].notna()

    final_df["rule_llm_conflict"] = (
        final_df["is_labeled"] &
        (final_df["rule_label"].astype("Int64") != final_df["label_retreat"])
    ).astype(int)

    def merge_reasons(row: pd.Series) -> str:
        """
        最小人工复核策略：
        - 不再因为 core_top10_manual_checked 就自动进复核
        - 不再因为一般 rule_conflict 就自动进复核
        - 只保留真正需要人工看的情况
        """
        if not row.get("is_labeled", False):
            return ""

        reasons = []

        raw_review_reason = str(row.get("review_reason", "") or "")
        raw_parts = [part for part in raw_review_reason.split("|") if part]

        # 只保留真正表示“人工仍需介入”的人工标记
        keep_direct = {
            "manual_override_top10",
            "manual_top10_needs_review",
        }
        for part in raw_parts:
            if part in keep_direct:
                reasons.append(part)

        # 低置信度才强制复核
        if pd.notna(row.get("confidence")) and row["confidence"] < 0.6:
            reasons.append("low_conf")

        # 只有正类却没有证据时，才强制人工看
        if (
            pd.notna(row.get("label_retreat")) and
            int(row["label_retreat"]) == 1 and
            not str(row.get("evidence_span", "")).strip()
        ):
            reasons.append("missing_evidence_positive")

        # 如果你后面想稍微更保守一点，可以打开这条：
        # if row.get("rule_llm_conflict", 0) == 1 and pd.notna(row.get("confidence")) and row["confidence"] < 0.8:
        #     reasons.append("rule_conflict_low_conf")

        # 去重保序
        seen = set()
        out = []
        for x in reasons:
            if x not in seen:
                seen.add(x)
                out.append(x)

        return "|".join(out)

    final_df["review_reason_final"] = final_df.apply(merge_reasons, axis=1)
    final_df["review_flag_final"] = final_df["review_reason_final"].apply(
        lambda value: "yes" if str(value).strip() else "no"
    )

    # 未标注样本不填 review_flag / review_reason
    final_df.loc[~final_df["is_labeled"], "review_flag_final"] = pd.NA
    final_df.loc[~final_df["is_labeled"], "review_reason_final"] = pd.NA

    label_columns = [
        "event_id",
        "text_id",
        "source",
        "term_id",
        "event_time_utc",
        "event_time_ny",
        "source_text",
        "rule_label",
        "rule_llm_conflict",
        "label_retreat",
        "confidence",
        "evidence_span",
        "review_flag_final",
        "review_reason_final",
        "feature_anchor_date",
        "target_trade_date",
        "key_event_id",
        "audit_tier",
        "model_name",
    ]

    labels_out = final_df[label_columns].rename(columns={
        "review_flag_final": "review_flag",
        "review_reason_final": "review_reason",
    })

    labels_out.to_csv(FINAL_LABEL_PATH, index=False)

    # QC 只基于已标注样本
    qc_base_df = final_df.loc[final_df["is_labeled"]].copy()

    overall_qc = {
        "segment": "overall",
        "term_id": "all",
        "source": "all",
        "n_events": len(qc_base_df),
        "low_conf_rate": float((qc_base_df["confidence"] < 0.6).fillna(False).mean()) if len(qc_base_df) else pd.NA,
        "rule_llm_conflict_rate": float(qc_base_df["rule_llm_conflict"].mean()) if len(qc_base_df) else pd.NA,
        "manual_review_rate": float(qc_base_df["review_flag_final"].eq("yes").mean()) if len(qc_base_df) else pd.NA,
        "manual_accept_rate": pd.NA,
    }

    if len(qc_base_df):
        by_group_qc = (
            qc_base_df
            .assign(
                low_conf=lambda df: (df["confidence"] < 0.6).fillna(False).astype(int),
                manual_review=lambda df: df["review_flag_final"].eq("yes").astype(int),
            )
            .groupby(["term_id", "source"], dropna=False)
            .agg(
                n_events=("event_id", "size"),
                low_conf_rate=("low_conf", "mean"),
                rule_llm_conflict_rate=("rule_llm_conflict", "mean"),
                manual_review_rate=("manual_review", "mean"),
            )
            .reset_index()
        )
        by_group_qc["segment"] = "term_source"
        by_group_qc["manual_accept_rate"] = pd.NA

        qc_df = pd.concat([
            pd.DataFrame([overall_qc]),
            by_group_qc[[
                "segment",
                "term_id",
                "source",
                "n_events",
                "low_conf_rate",
                "rule_llm_conflict_rate",
                "manual_review_rate",
                "manual_accept_rate",
            ]],
        ], ignore_index=True)
    else:
        qc_df = pd.DataFrame([overall_qc])

    qc_df.to_csv(QC_AUDIT_PATH, index=False)

    print(f"Wrote final labels: {FINAL_LABEL_PATH}")
    print(f"Wrote QC audit: {QC_AUDIT_PATH}")
    print(f"Labeled rows in final output: {labels_out['label_retreat'].notna().sum():,}")

    labels_out.head(10)


Wrote final labels: /Users/horace/Coding/ML finance/project/data/processed/standardized/labels_mvp_v1.csv
Wrote QC audit: /Users/horace/Coding/ML finance/project/data/manual/label_qc_audit_v1.csv
Labeled rows in final output: 1,886
